In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
import copy
import pickle

import numpy as np
import pandas as pd
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

from sklearn.metrics import mean_squared_error, r2_score, make_scorer

import deepchem as dc

In [ ]:
def display_df(df):
    display(HTML(df.to_html()))
    return None

In [ ]:
def rmse(estimator, X_eval, y_eval):
    y_hat = estimator.predict(X_eval)
    return np.sqrt(mean_squared_error(y_eval.flatten(), y_hat.flatten()))


def r2(estimator, X_eval, y_eval):
    y_hat = estimator.predict(X_eval)
    return r2_score(y_eval.flatten(), y_hat.flatten())


def peason_r(estimator, X_eval, y_eval):
    y_hat = estimator.predict(X_eval)
    return np.corrcoef(y_eval.flatten(), y_hat.flatten())[0, 1]


def peason_r_metric(y_true, y_pred):
    return np.corrcoef(y_true.flatten(), y_pred.flatten())[0, 1]

peason_r_score = make_scorer(peason_r_metric)

In [ ]:
def check_smiles_in(simles, smiles_lst):
    assert isinstance(simles, str)
    assert isinstance(smiles_lst, list)
    return simles.lower().strip() in [s.lower().strip() for s in smiles_lst]

# Create train/test datasets

In [ ]:
pdXY = pd.read_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_109ft_imputed_std.csv")
SMILES_TRAIN_TEST = pdXY["smiles"].unique().tolist()
print("pdXY", pdXY.shape)
PDY_COLS = ["new_id", "smiles", "dG", "code", "train_test", "smiles_len"]
pdXY = pdXY[PDY_COLS]
print("pdXY", pdXY.shape)


ii = pdXY["train_test"] == "train"
pdXY[ii].to_csv("data/process/pdY_train.csv", index=False)

jj = pdXY["train_test"] == "test"
pdXY[jj].to_csv("data/process/pdY_test.csv", index=False)

In [ ]:
train_file = "data/process/pdY_train.csv"
test_file = "data/process/pdY_test.csv"

tasks = ["dG"]
ntasks = len(tasks)
featurizer_func = dc.feat.ConvMolFeaturizer()
loader = dc.data.CSVLoader(tasks=tasks, feature_field='smiles', featurizer=featurizer_func)

train_dataset = loader.create_dataset(train_file)
test_dataset = loader.create_dataset(test_file)


## Create chembl36excl33 datasets

In [ ]:
assert False
pdXY = pd.read_csv("data/process/pdXY_chembl36excl33_rdkit_descriptors_109ft_imputed_std.csv")
print("pdXY", pdXY.shape)
pdXY = pdXY[PDY_COLS]
print("pdXY", pdXY.shape)

pdXY.to_csv("data/process/pdY_chembl36excl33_dataset.csv", index=False)

In [ ]:
data_file = "data/process/pdY_chembl36excl33_dataset.csv"
data_dir="data/process/graph_features/chembl36excl33"

pdY = pd.read_csv(data_file)
print(pdY.shape)

tasks = ["dG"]
ntasks = len(tasks)
featurizer_func = dc.feat.ConvMolFeaturizer()
loader = dc.data.CSVLoader(tasks=tasks, feature_field='smiles', featurizer=featurizer_func)

dataset = loader.create_dataset(inputs=[data_file,], data_dir=data_dir)
#dataset = dc.data.DiskDataset(data_dir="data/process/graph_features/chembl36excl33")

## create amylovis201 data set

In [ ]:
assert False
pdXY = pd.read_csv("data/process/pdXY_amylovis201_rdkit_descriptors_109ft_imputed_std.csv")
print("pdXY", pdXY.shape)
pdXY = pdXY[PDY_COLS]
print("pdXY", pdXY.shape)

pdXY.to_csv("data/process/pdY_amylovis201_dataset.csv", index=False)

In [ ]:
data_file = "data/process/pdY_amylovis201_dataset.csv"
data_dir="data/process/graph_features/amylovis201"

pdY = pd.read_csv(data_file)
print(pdY.shape)

tasks = ["dG"]
ntasks = len(tasks)
featurizer_func = dc.feat.ConvMolFeaturizer()
loader = dc.data.CSVLoader(tasks=tasks, feature_field='smiles', featurizer=featurizer_func)

dataset = loader.create_dataset(inputs=[data_file,], data_dir=data_dir)
#dataset = dc.data.DiskDataset(data_dir="data/process/graph_features/amylovis201")

## create ref01 data set

In [ ]:
assert True
pdXY = pd.read_csv("data/process/pdXY_ref01_rdkit_descriptors_109ft_imputed_std.csv")
print("pdXY", pdXY.shape)
pdXY = pdXY[PDY_COLS]
print("pdXY", pdXY.shape)

pdXY.to_csv("data/process/pdY_ref01_dataset.csv", index=False)

In [ ]:
data_file = "data/process/pdY_ref01_dataset.csv"
data_dir="data/process/graph_features/ref01"

pdY = pd.read_csv(data_file)
print(pdY.shape)

tasks = ["dG"]
ntasks = len(tasks)
featurizer_func = dc.feat.ConvMolFeaturizer()
loader = dc.data.CSVLoader(tasks=tasks, feature_field='smiles', featurizer=featurizer_func)

dataset = loader.create_dataset(inputs=[data_file,], data_dir=data_dir)
#dataset = dc.data.DiskDataset(data_dir="data/process/graph_features/ref01")

## Default

In [ ]:
model_dir="models/graph/gconv_default"

model = dc.models.GraphConvModel(ntasks, mode='regression', model_dir=model_dir)
model.fit(train_dataset, nb_epoch=50)

print("Train RMSE:", rmse(model, train_dataset, train_dataset.y))
print("Train Pearson's R:", peason_r(model, train_dataset, train_dataset.y))

print("Test RMSE:", rmse(model, test_dataset, test_dataset.y))
print("Test Pearson's R:", peason_r(model, test_dataset, test_dataset.y))

In [ ]:
model = dc.models.GraphConvModel(ntasks, mode='regression', model_dir=model_dir)
model.restore()

print("Train RMSE:", rmse(model, train_dataset, train_dataset.y))
print("Train Pearson's R:", peason_r(model, train_dataset, train_dataset.y))

print("Test RMSE:", rmse(model, test_dataset, test_dataset.y))
print("Test Pearson's R:", peason_r(model, test_dataset, test_dataset.y))

## `graph_conv_layers = [64, 64, 64]`, `dense_layer_size=128`, `dropout=0`

In [ ]:
model_dir = "models/graph/gconv_01"

batch_size = 64
batches_per_epoch = 900/batch_size
learning_rate = dc.models.optimizers.ExponentialDecay(0.0001, 0.9, batches_per_epoch)

graph_conv_layers = [64, 64, 64]
dense_layer_size = 128
dropout = 0.
model = dc.models.GraphConvModel(ntasks, 
                                 graph_conv_layers=graph_conv_layers, 
                                 dense_layer_size=dense_layer_size,
                                 dropout=dropout,
                                 batch_normalize=True,
                                 mode='regression', 
                                 batch_size=batch_size, 
                                 learning_rate=learning_rate,
                                 model_dir=model_dir)

metric = dc.metrics.Metric(dc.metrics.rms_score)
callback = dc.models.ValidationCallback(test_dataset, 100, metric)
model.fit(train_dataset, nb_epoch=100, callbacks=callback)

print("Train RMSE:", rmse(model, train_dataset, train_dataset.y))
print("Train Pearson's R:", peason_r(model, train_dataset, train_dataset.y))

print("Test RMSE:", rmse(model, test_dataset, test_dataset.y))
print("Test Pearson's R:", peason_r(model, test_dataset, test_dataset.y))

## `graph_conv_layers = [64, 64, 64]`, `dense_layer_size=64`, `dropout=0.1`

In [ ]:
!rm -r models/graph/gconv_02
model_dir = "models/graph/gconv_02"

batch_size = 32
batches_per_epoch = 640/batch_size
learning_rate = dc.models.optimizers.ExponentialDecay(0.0001, 0.9, batches_per_epoch)

graph_conv_layers = [64, 64, 64]
dense_layer_size = 64
dropout = 0.1
model = dc.models.GraphConvModel(ntasks, 
                                 graph_conv_layers=graph_conv_layers, 
                                 dense_layer_size=dense_layer_size,
                                 dropout=dropout,
                                 batch_normalize=True,
                                 mode='regression', 
                                 batch_size=batch_size, 
                                 learning_rate=learning_rate,
                                 model_dir=model_dir)

metric = dc.metrics.Metric(dc.metrics.rms_score)
callback = dc.models.ValidationCallback(test_dataset, 100, metric)
model.fit(train_dataset, nb_epoch=100, callbacks=callback)

print("Train RMSE:", rmse(model, train_dataset, train_dataset.y))
print("Train Pearson's R:", peason_r(model, train_dataset, train_dataset.y))

print("Test RMSE:", rmse(model, test_dataset, test_dataset.y))
print("Test Pearson's R:", peason_r(model, test_dataset, test_dataset.y))

## `graph_conv_layers = [64, 64, 128]`, `dense_layer_size=128`, `dropout=0.1`

In [ ]:
assert False
model_dir = "models/graph/gconv_03"

batch_size = 32
batches_per_epoch = 600/batch_size
learning_rate = dc.models.optimizers.ExponentialDecay(0.0001, 0.9, batches_per_epoch)

graph_conv_layers = [64, 64, 128]
dense_layer_size = 128
dropout = 0.1
model = dc.models.GraphConvModel(ntasks, 
                                 graph_conv_layers=graph_conv_layers, 
                                 dense_layer_size=dense_layer_size,
                                 dropout=dropout,
                                 batch_normalize=True,
                                 mode='regression', 
                                 batch_size=batch_size, 
                                 learning_rate=learning_rate,
                                 model_dir=model_dir)

metric = dc.metrics.Metric(dc.metrics.rms_score)
callback = dc.models.ValidationCallback(test_dataset, 100, metric)
model.fit(train_dataset, nb_epoch=100, callbacks=callback)

print("Train RMSE:", rmse(model, train_dataset, train_dataset.y))
print("Train Pearson's R:", peason_r(model, train_dataset, train_dataset.y))

print("Test RMSE:", rmse(model, test_dataset, test_dataset.y))
print("Test Pearson's R:", peason_r(model, test_dataset, test_dataset.y))

## `graph_conv_layers = [64, 64, 128]`, `dense_layer_size=128`, `dropout=0.1`

In [ ]:

model_dir = "models/graph/gconv_04"

batch_size = 32
batches_per_epoch = 640/batch_size
learning_rate = dc.models.optimizers.ExponentialDecay(0.0001, 0.9, batches_per_epoch)

graph_conv_layers = [64, 64, 128]
dense_layer_size = 128
dropout = 0.0
model = dc.models.GraphConvModel(ntasks, 
                                 graph_conv_layers=graph_conv_layers, 
                                 dense_layer_size=dense_layer_size,
                                 dropout=dropout,
                                 batch_normalize=True,
                                 mode='regression', 
                                 batch_size=batch_size, 
                                 learning_rate=learning_rate,
                                 model_dir=model_dir)

metric = dc.metrics.Metric(dc.metrics.rms_score)
callback = dc.models.ValidationCallback(test_dataset, 100, metric)
model.fit(train_dataset, nb_epoch=100, callbacks=callback)

print("Train RMSE:", rmse(model, train_dataset, train_dataset.y))
print("Train Pearson's R:", peason_r(model, train_dataset, train_dataset.y))

print("Test RMSE:", rmse(model, test_dataset, test_dataset.y))
print("Test Pearson's R:", peason_r(model, test_dataset, test_dataset.y))

## chose `models/graph/gconv_03`

### predict for train

In [ ]:
csv_file = "data/process/pdY_train.csv"
pdY = pd.read_csv(csv_file)
print(pdY.shape)

tasks = ["dG"]
ntasks = len(tasks)
featurizer_func = dc.feat.ConvMolFeaturizer()
loader = dc.data.CSVLoader(tasks=tasks, feature_field='smiles', featurizer=featurizer_func)
dataset = loader.create_dataset(csv_file)

model_dir = "models/graph/gconv_03"

batch_size = 32
batches_per_epoch = 600/batch_size
learning_rate = dc.models.optimizers.ExponentialDecay(0.0001, 0.9, batches_per_epoch)

graph_conv_layers = [64, 64, 128]
dense_layer_size = 128
dropout = 0.1
model = dc.models.GraphConvModel(ntasks, 
                                 graph_conv_layers=graph_conv_layers, 
                                 dense_layer_size=dense_layer_size,
                                 dropout=dropout,
                                 batch_normalize=True,
                                 mode='regression', 
                                 batch_size=batch_size, 
                                 learning_rate=learning_rate,
                                 model_dir=model_dir)


model.restore()
ypred = model.predict(dataset)
df00 = pd.DataFrame({"smiles": dataset.ids, "dG": dataset.y[:,0], "pred": ypred[:,0]})
df00.to_csv("results/graph/train.csv", index=False)

## predict for test

In [ ]:
csv_file = "data/process/pdY_test.csv"
pdY = pd.read_csv(csv_file)
print(pdY.shape)

tasks = ["dG"]
ntasks = len(tasks)
featurizer_func = dc.feat.ConvMolFeaturizer()
loader = dc.data.CSVLoader(tasks=tasks, feature_field='smiles', featurizer=featurizer_func)
dataset = loader.create_dataset(csv_file)

model_dir = "models/graph/gconv_03"

batch_size = 32
batches_per_epoch = 600/batch_size
learning_rate = dc.models.optimizers.ExponentialDecay(0.0001, 0.9, batches_per_epoch)

graph_conv_layers = [64, 64, 128]
dense_layer_size = 128
dropout = 0.1
model = dc.models.GraphConvModel(ntasks, 
                                 graph_conv_layers=graph_conv_layers, 
                                 dense_layer_size=dense_layer_size,
                                 dropout=dropout,
                                 batch_normalize=True,
                                 mode='regression', 
                                 batch_size=batch_size, 
                                 learning_rate=learning_rate,
                                 model_dir=model_dir)


model.restore()


model.restore()
ypred = model.predict(dataset)
df00 = pd.DataFrame({"smiles": dataset.ids, "dG": dataset.y[:,0], "pred": ypred[:,0]})
df00.to_csv("results/graph/test.csv", index=False)

## predict for chembl33

In [ ]:
!ls /home/hai/data/Compound_Screen/GSK3b/data/process/graph_features/chembl33

In [ ]:
assert False
dataset = dc.data.DiskDataset(data_dir="/home/hai/data/Compound_Screen/GSK3b/data/process/graph_features/chembl33")
tasks = ["dG"]
ntasks = len(tasks)

model_dir = "models/graph/gconv_03"

batch_size = 32
batches_per_epoch = 600/batch_size
learning_rate = dc.models.optimizers.ExponentialDecay(0.0001, 0.9, batches_per_epoch)

graph_conv_layers = [64, 64, 128]
dense_layer_size = 128
dropout = 0.1
model = dc.models.GraphConvModel(ntasks, 
                                 graph_conv_layers=graph_conv_layers, 
                                 dense_layer_size=dense_layer_size,
                                 dropout=dropout,
                                 batch_normalize=True,
                                 mode='regression', 
                                 batch_size=batch_size, 
                                 learning_rate=learning_rate,
                                 model_dir=model_dir)

model.restore()
ypred = model.predict(dataset)
df00 = pd.DataFrame({"smiles": dataset.ids, "dG": dataset.y[:,0], "pred": ypred[:,0]})
print("df00", df00.shape)

df00.to_csv("results/graph/chembl33.csv", index=False)

In [ ]:
df00.sort_values(by="pred").head(1000)

## predict for chembl36excl33

In [ ]:
assert False
dataset = dc.data.DiskDataset(data_dir="data/process/graph_features/chembl36excl33")
tasks = ["dG"]
ntasks = len(tasks)

model_dir = "models/graph/gconv_03"

batch_size = 32
batches_per_epoch = 600/batch_size
learning_rate = dc.models.optimizers.ExponentialDecay(0.0001, 0.9, batches_per_epoch)

graph_conv_layers = [64, 64, 128]
dense_layer_size = 128
dropout = 0.1
model = dc.models.GraphConvModel(ntasks, 
                                 graph_conv_layers=graph_conv_layers, 
                                 dense_layer_size=dense_layer_size,
                                 dropout=dropout,
                                 batch_normalize=True,
                                 mode='regression', 
                                 batch_size=batch_size, 
                                 learning_rate=learning_rate,
                                 model_dir=model_dir)

model.restore()
ypred = model.predict(dataset)
df00 = pd.DataFrame({"smiles": dataset.ids, "dG": dataset.y[:,0], "pred": ypred[:,0]})
print("df00", df00.shape)

df00.to_csv("results/graph/chembl36excl33.csv", index=False)

In [ ]:
df00.sort_values(by="pred").head(100)

## predict for amylovis201

In [ ]:
assert False
dataset = dc.data.DiskDataset(data_dir="data/process/graph_features/amylovis201")
tasks = ["dG"]
ntasks = len(tasks)

model_dir = "models/graph/gconv_03"

batch_size = 32
batches_per_epoch = 600/batch_size
learning_rate = dc.models.optimizers.ExponentialDecay(0.0001, 0.9, batches_per_epoch)

graph_conv_layers = [64, 64, 128]
dense_layer_size = 128
dropout = 0.1
model = dc.models.GraphConvModel(ntasks, 
                                 graph_conv_layers=graph_conv_layers, 
                                 dense_layer_size=dense_layer_size,
                                 dropout=dropout,
                                 batch_normalize=True,
                                 mode='regression', 
                                 batch_size=batch_size, 
                                 learning_rate=learning_rate,
                                 model_dir=model_dir)

model.restore()
ypred = model.predict(dataset)
df00 = pd.DataFrame({"smiles": dataset.ids, "dG": dataset.y[:,0], "pred": ypred[:,0]})
print("df00", df00.shape)

df00.to_csv("results/graph/amylovis201.csv", index=False)

In [ ]:
df00

## predict for ref01

In [ ]:
assert True
dataset = dc.data.DiskDataset(data_dir="data/process/graph_features/ref01")
tasks = ["dG"]
ntasks = len(tasks)

model_dir = "models/graph/gconv_03"

batch_size = 32
batches_per_epoch = 600/batch_size
learning_rate = dc.models.optimizers.ExponentialDecay(0.0001, 0.9, batches_per_epoch)

graph_conv_layers = [64, 64, 128]
dense_layer_size = 128
dropout = 0.1
model = dc.models.GraphConvModel(ntasks, 
                                 graph_conv_layers=graph_conv_layers, 
                                 dense_layer_size=dense_layer_size,
                                 dropout=dropout,
                                 batch_normalize=True,
                                 mode='regression', 
                                 batch_size=batch_size, 
                                 learning_rate=learning_rate,
                                 model_dir=model_dir)

model.restore()
ypred = model.predict(dataset)
df00 = pd.DataFrame({"smiles": dataset.ids, "dG": dataset.y[:,0], "pred": ypred[:,0]})
print("df00", df00.shape)

df00.to_csv("results/graph/ref01.csv", index=False)

In [ ]:
df00